# dcgan-normal-init-002 — ex2: named-module DCGAN init with per-type gain and a report

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dcgan-normal-init-002`. Running the final beacon cell reports progress against the `GAN: DCGAN normal init 0.02` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: DCGAN normal init 0.02` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dcgan-normal-init-002`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dcgan-normal-init-002"
DD_SUBTOPIC = "GAN: DCGAN normal init 0.02"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## DCGAN normal init — named-module variant

Ex1 walked layers via `model.apply` (a pure transform).  Here we want a **report**: which named submodule got which init, by dotted path. The tool is `model.named_modules()` — same recursion as `modules()`, but each yielded item is `(qualified_name: str, module)`:

```python
for name, m in model.named_modules():
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, mean=0.0, std=0.02)
        records.append((name, 'conv', m.weight.std().item()))
```

**Why named_modules over modules.** When you need to LOG what you initialized (which is most real-world training-script init code), you want the qualified name (`'features.3.conv'`) — not just the type. `apply` gives you neither.

**Why also gain-scale.** Production DCGAN code sometimes scales the std by a `gain` factor per layer type (e.g. ConvTranspose gets `gain` × 0.02 to compensate for the upsample). The ex2 drills the parametric form.

### Exercise 2 — named-module DCGAN init with per-type gain and a report

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a model by walking `model.named_modules()`, apply gain-scaled `N(0, gain*0.02)` init to Conv/ConvTranspose layers (gain per type), and return a sorted report of `(qualified_name, type_label, post_init_std)`.
> Keywords: dcgan, init, named-modules, gain, report
> ```

**KCs targeted:** `named-modules-iter-with-qname`, `gain-scaled-normal-init`

Implement `ex2_named_dcgan_init(model, conv_gain=1.0, convt_gain=1.0)`. Three responsibilities:

1. Walk `model.named_modules()`. SKIP the root entry (qname == `''`).
2. For each yielded `(qname, m)`:
   - If `isinstance(m, nn.Conv2d)`: call `nn.init.normal_(m.weight, 0.0, conv_gain * 0.02)`, then append `(qname, 'conv2d', m.weight.std().item())` to a `records` list.
   - elif `isinstance(m, nn.ConvTranspose2d)`: call `nn.init.normal_(m.weight, 0.0, convt_gain * 0.02)`, then append `(qname, 'convtranspose2d', m.weight.std().item())`.
   - Other layer types: skip (no init, no record).
3. Return `sorted(records)` — sorted by qname (lexicographic, default tuple sort).

Input: `model` — `nn.Module`; `conv_gain`, `convt_gain` — floats, default 1.0.
Output: `list[tuple[str, str, float]]`.

The visualization plots the per-layer post-init std as a bar chart with two colors (conv vs convt), showing how the gain knobs shift the std away from the baseline 0.02.

In [ ]:
def ex2_named_dcgan_init(model: nn.Module, conv_gain: float = 1.0, convt_gain: float = 1.0) -> list:
    records = []
    for qname, m in model.named_modules():
        if qname == '':
            continue
        if isinstance(m, nn.Conv2d):
            nn.init.normal_(m.weight, 0.0, conv_gain * 0.02)
            records.append((qname, 'conv2d', m.weight.std().item()))
        elif isinstance(m, nn.ConvTranspose2d):
            nn.init.normal_(m.weight, 0.0, convt_gain * 0.02)
            records.append((qname, 'convtranspose2d', m.weight.std().item()))
    return sorted(records)


<details><summary>Solution</summary>

```python
def ex2_named_dcgan_init(model: nn.Module, conv_gain: float = 1.0, convt_gain: float = 1.0) -> list:
    records = []
    for qname, m in model.named_modules():
        if qname == '':
            continue
        if isinstance(m, nn.Conv2d):
            nn.init.normal_(m.weight, 0.0, conv_gain * 0.02)
            records.append((qname, 'conv2d', m.weight.std().item()))
        elif isinstance(m, nn.ConvTranspose2d):
            nn.init.normal_(m.weight, 0.0, convt_gain * 0.02)
            records.append((qname, 'convtranspose2d', m.weight.std().item()))
    return sorted(records)
```

**`named_modules()` over `apply`.** `apply` gives you the module but not its name — fine for pure init, bad for logging. The named form is the canonical pattern in training scripts that emit per-layer stats to wandb or tensorboard.

**Filter root via `qname == ''`.** The first item from `named_modules()` is always the root model — empty qname, model itself as the module. Forgetting to skip it doesn't break the init (the model itself isn't a Conv2d) but it appears as a stray in any qname-keyed dict.

**Gain as a multiplier, not a replacement.** Multiplying 0.02 by a gain factor lets you keep the DCGAN base std while tuning per type. Common in production: `gain=1.4` on the final ConvT to compensate for the Tanh activation.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()